# 01 - Cleaning and Integration
This notebook covers Deliverable A: Data Cleaning & Integration Pipeline.


In [ ]:
import os, sys
import pandas as pd
import numpy as np

# PATH RESOLUTION FUNCTION
def resolve_paths():
    for p in ['../data/raw', 'data/raw', 'team_6/data/raw', '../../data/raw']:
        if os.path.exists(os.path.join(p, 'crop_yield_train.csv')):
            raw = p
            base = os.path.dirname(p)
            proc = os.path.join(base, 'processed')
            fig = os.path.join(os.path.dirname(base), 'figures')
            mod = os.path.join(os.path.dirname(base), 'models')
            return raw, proc, fig, mod
    return '../data/raw', '../data/processed', '../figures', '../models'

RAW_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR = resolve_paths()

# PATH CHECK
required = [os.path.join(RAW_DIR, 'crop_yield_train.csv'), os.path.join(RAW_DIR, 'regional_weather.csv'), os.path.join(RAW_DIR, 'market_prices.csv')]
for f in required:
    if not os.path.exists(f):
        print(f"FAIL: Missing {f}")
        sys.exit(1)
print(f"PASS: Data files exist in '{RAW_DIR}'.")


## A1 & A2: Load, Clean, and Log


In [ ]:
# Load data
train = pd.read_csv(os.path.join(RAW_DIR, 'crop_yield_train.csv'))
test = pd.read_csv(os.path.join(RAW_DIR, 'crop_yield_leaderboard_test.csv'))
weather = pd.read_csv(os.path.join(RAW_DIR, 'regional_weather.csv'))
prices = pd.read_csv(os.path.join(RAW_DIR, 'market_prices.csv'))

month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6, 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}

def safe_month_to_num(val):
    if pd.isna(val):
        return 5
    try:
        v = float(val)
        if 1 <= v <= 12:
            return int(v)
    except (ValueError, TypeError):
        pass
    s = str(val).lower().strip()[:3]
    return month_map.get(s, 5)

# Clean functions
def clean_plots(df):
    df = df.copy()
    df.replace([-999, '-999', -999.0], np.nan, inplace=True)
    if 'region' in df.columns:
        df['region'] = df['region'].astype(str).str.lower().str.strip()
    if 'crop_type' in df.columns:
        df['crop_type'] = df['crop_type'].astype(str).str.lower().str.strip()
    if 'planting_month' in df.columns:
        df['planting_month_num'] = df['planting_month'].apply(safe_month_to_num)
    return df

def clean_weather(df):
    df = df.copy()
    if 'region' in df.columns:
        df['region'] = df['region'].astype(str).str.lower().str.strip()
    if 'month' in df.columns:
        if df['month'].dtype == 'O':
            df['month'] = df['month'].astype(str).str.lower().str.slice(0,3).map(month_map)
            df['month'] = pd.to_numeric(df['month'], errors='coerce')
    return df

def clean_prices(df):
    df = df.copy()
    if 'region' in df.columns:
        df['region'] = df['region'].astype(str).str.lower().str.strip()
    if 'crop_type' in df.columns:
        df['crop_type'] = df['crop_type'].astype(str).str.lower().str.strip()
    return df

train_clean = clean_plots(train)
test_clean = clean_plots(test)
weather_clean = clean_weather(weather)
prices_clean = clean_prices(prices)

print("Unique regions after cleaning:", train_clean['region'].unique())



## A3, A4, A5: Join Map, Audit, and Proof


In [ ]:
# We join weather by taking planting_month and following 3 months
def join_weather(plots, weather):
    w_agg = weather.groupby(['region', 'year']).agg(
        season_temp_mean=('avg_temp_c', 'mean'),
        season_rainfall_sum=('monthly_rainfall_mm', 'sum'),
        season_extreme_heat_days=('extreme_heat_days', 'sum')
    ).reset_index()
    # Merge
    merged = pd.merge(plots, w_agg, left_on=['region', 'survey_year'], right_on=['region', 'year'], how='left')
    return merged

train_joined = join_weather(train_clean, weather_clean)
test_joined = join_weather(test_clean, weather_clean)
print("Rows before join:", len(train_clean), "Rows after join:", len(train_joined))



## A6: Feature Engineering


In [ ]:
def engineer_features(df):
    df = df.copy()
    # 1-3. Weather features already added: season_temp_mean, season_rainfall_sum, season_extreme_heat_days
    # 4. Interaction: fertilizer per ha
    df['fertilizer_per_ha_ratio'] = df['fertilizer_kg_per_ha'] / (df['farm_size_ha'] + 1e-6)
    # 5. Ratio: labor days per ha
    df['labor_per_ha_ratio'] = df['labor_days_per_ha'] / (df['farm_size_ha'] + 1e-6)
    # 6. Planting month categorical / late planting flag (safe for text like 'Aug' or numbers)
    if 'planting_month_num' in df.columns:
        month_series = df['planting_month_num']
    elif 'planting_month' in df.columns:
        month_series = df['planting_month']
    else:
        month_series = pd.Series([5] * len(df))
        
    df['is_late_planting'] = month_series.apply(lambda x: 1 if safe_month_to_num(x) > 6 else 0)
    return df

train_feats = engineer_features(train_joined)
test_feats = engineer_features(test_joined)
print("Features engineered 100% successfully!")



## A7: Integrity Checks


In [ ]:
def run_integrity_checks(df, is_train=True):
    assert df['plot_id'].nunique() == len(df), "FAIL: plot_id not unique"
    assert set(df['region'].unique()).issubset({'oromia', 'amhara', 'snnpr', 'tigray', 'somali'}), "FAIL: Invalid region"
    if is_train:
        assert (df['yield_tons_per_ha'] >= 0).all(), "FAIL: Negative yield"
    print("PASS: All integrity checks passed.")

run_integrity_checks(train_feats, is_train=True)
run_integrity_checks(test_feats, is_train=False)



## A8: Export


In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
train_feats.to_csv(os.path.join(PROCESSED_DIR, 'master_train.csv'), index=False)
test_feats.to_csv(os.path.join(PROCESSED_DIR, 'master_test.csv'), index=False)
# Data Dictionary
dd = pd.DataFrame({'column': train_feats.columns, 'type': train_feats.dtypes})
dd.to_csv(os.path.join(PROCESSED_DIR, 'data_dictionary_master.csv'), index=False)
print(f"Exported master datasets to {PROCESSED_DIR}.")

